# GSD — All7 crop560 + Gaussian blur/JPEG, fixed validation

Gắn **yangsangtai/tiny-genimage** VÀ dataset/output chứa manifest của lần train All7 trước.
Bật GPU và Internet. Notebook tự chứa toàn bộ code.
Đây là thí nghiệm MỚI; không thay thế notebook/checkpoint raw-only.

## Input và chế độ bỏ scan
Mặc định `data_mode='reuse_manifests'`: không decode/hash toàn bộ ảnh, không chia lại split.
Input thứ hai cần thư mục sau (không cần best.pt/last.pt):

```text
<manifest_root>/
  config.json                    # khuyến nghị, nếu có
  seed42/
    train_manifest.csv           # bắt buộc
    validation_manifest.csv      # bắt buộc
    test_manifest.csv            # bắt buộc
    split_summary.json           # khuyến nghị để đối chiếu thứ tự/count
```

Có thể dùng `hadinho0710/checkpoint-gsd-sdv5-train` phiên bản chứa
`gsd_raw_all7_crop560_v1/seed42/` hoặc upload gói manifest-only đi kèm.
Để `manifest_root=None` và `dataset_root=None` cho tự tìm nếu chỉ có một ứng viên;
nếu có nhiều, điền chính xác đường dẫn thư mục tương ứng dưới /kaggle/input.
Đường dẫn tuyệt đối cũ trong CSV được bỏ qua; code map lại bằng relative_path.
Chỉ kiểm tra metadata/ảnh trùng theo hash ĐÃ LƯU và file tồn tại/đúng byte length.
Tin tưởng audit trước và đúng phiên bản dataset: không phát hiện thay đổi nội dung cùng kích thước file,
không phát hiện ảnh mới ngoài manifest. Thiếu/sai input thì dừng, KHÔNG tự fallback sang full scan.
Muốn audit lại phải chủ động đặt `data_mode='full_scan'`, dùng output_root mới.

## Train
Giữ All7, center crop tối đa 560 mỗi chiều, edge-pad14, token động, logical batch8,
microbatch1, CLIP/GSD và optimizer cũ. Không resize, flip, consistency loss hay model phụ.
JPEG và blur thực hiện trên RGB gốc TRƯỚC crop/padding/normalization:

- Blur: xác suất 0.5, sigma lấy đều trong [0, 2].
- JPEG: xác suất 0.5 độc lập, quality nguyên lấy đều 75–100, subsampling0 (4:4:4).
- Nếu cùng được chọn: blur → JPEG. Trung bình bốn nhóm native/blur/JPEG/cả hai: 25% mỗi nhóm.
- Cùng phân phối cho real/fake và mọi generator; lấy lại tham số mỗi epoch.
- Anchor đóng băng và detector nhận CÙNG tensor đã xử lý; không tăng số forward train.

Paper GSD có JPEG/blur nhưng không công bố đầy đủ recipe. Các xác suất/khoảng trên là lựa
chọn triển khai, KHÔNG phải tham số chính thức. All7/crop560/batch8 cũng khác protocol paper SD14.
Nguồn: https://arxiv.org/html/2603.09242v2#S5.SS1

## Validation và early stopping
Chỉ dùng validation tách từ TRAIN folders theo pixel-hash groups như cũ.
Official test không dùng chọn checkpoint. Mỗi epoch đánh giá hai view cùng ID/order/batch:

1. Native: không blur/JPEG thêm.
2. Processed: cùng phân phối augmentation train, nhưng tham số theo ảnh được cố định bằng seed.

`val_loss = (val_native_loss + val_processed_loss) / 2`, tính CE trực tiếp từ logits.
Early stopping patience3/min_delta0.001, tối đa50 epoch; khôi phục best.pt theo loss này.
`val_native` và `val_processed` ghi riêng accuracy/AUC/AP/recall/confusion matrix tại threshold0.5.
Validation mất thêm một lượt inference; không random lại processing giữa các epoch.
Ảnh validation không lưu thêm: bảng tham số CSV đủ tái tạo bằng code/seed/config.

## Chạy an toàn
Khởi tạo từ pretrained CLIP. Không gắn best.pt raw-only làm điểm khởi tạo.
`resume=True` CHỈ khôi phục last.pt của chính giao thức mới, cùng config/manifests.
Chạy smoke trong output_root RIÊNG trước; sau đó full run ở thư mục mặc định khác.
Không tự giảm ảnh/batch, không bỏ ảnh khi OOM. Chưa kiểm chứng full-size CUDA trên Kaggle.
Chế độ mặc định chỉ kiểm tra manifest/path/size khi chạy lại; resume cuối epoch, không giữa epoch.
Manifest có thể lấy từ run raw-only nhưng KHÔNG lấy trọng số detector raw-only để tiếp tục train.

Output: `/kaggle/working/gsd_all7_crop560_blur_jpeg_manifest_v2`.
Sau mỗi epoch: best.pt/last.pt, history.json, validation_metrics.csv,
validation_predictions_epochNNN.csv. Mỗi seed có validation_processing_plan.csv và validation_protocol.json.
Lưu/download output của Kaggle trước khi reset session; /kaggle/working không phải lưu trữ vĩnh viễn.


In [ ]:
%pip install -q "transformers==4.46.3" "huggingface_hub>=0.23.2,<1.0" "pandas<3" "scikit-learn<1.9" "pillow<12" matplotlib tqdm


## Runtime
Không cần upload file Python riêng.

In [ ]:
"""Paired full-CLIP/GSD experiment. Self-contained Kaggle runtime.

Paper v2 Fig.3: GSD is BEFORE each of the last four transformer blocks.
No fake-only codec transformations, test tuning, or automatic batch-size changes.
"""
from __future__ import annotations

import copy
import gc
import hashlib
import io
import json
import math
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageFilter, ImageOps
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix, average_precision_score
from transformers import CLIPVisionModel, CLIPVisionConfig
from tqdm.auto import tqdm


DEFAULT_CONFIG = dict(
    dataset_root=None, input_root='/kaggle/input', output_root='/kaggle/working/clip_gsd_tiny',
    model_id='openai/clip-vit-large-patch14', local_files_only=False,
    seeds=[42], models=['clip_gsd'], epochs=50,
    early_stopping=True, patience=3, min_delta=0.001,
    train_batch=128, test_batch=512, micro_batch=2, anchor_micro_batch=8,
    lr=1e-6, head_lr=1e-6, modulation_lr=1e-6, weight_decay=0.01,
    betas=[0.9, 0.999], eps=1e-8, k=64, lambda_max=1., a_init=2., b_init=1.,
    validation_fraction=0.1, num_workers=2, gradient_checkpointing=True,
    precision='fp16', svd_driver='gesvdj', resume=True,
    jpeg_probability=0.5, jpeg_train_quality=[75, 100], jpeg_subsampling=0,
    blur_probability=0.5, blur_sigma=[0., 2.], horizontal_flip_probability=0.5,
    jpeg_eval_qualities=[96], size_control=256, threshold=0.5,
    pooling='patch_mean_post_ln', source_generator='sd15', strict_eight_generators=False,
    expected_generators=['sd15', 'adm', 'biggan', 'glide', 'midjourney', 'wukong', 'vqdm'],
    smoke=False, smoke_train_per_class=64, smoke_eval_per_class=4,
)


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def canonical_generator(name):
    n = ''.join(c for c in name.lower() if c.isalnum())
    aliases = {'sd14': ('sd14', 'sdv14', 'sdv4', 'stablediffusionv14', 'stablediffusion14'),
               'sd15': ('sd15', 'sdv15', 'sdv5', 'stablediffusionv15', 'stablediffusion15'),
               'adm': ('adm', 'guided'), 'biggan': ('biggan',), 'glide': ('glide',),
               'midjourney': ('midjourney',), 'wukong': ('wukong',), 'vqdm': ('vqdm',)}
    matches = [key for key, values in aliases.items() if any(v in n for v in values)]
    return matches[0] if len(matches) == 1 else None


def find_root(cfg):
    if cfg['dataset_root']:
        return Path(cfg['dataset_root'])
    roots = set()
    for p in Path(cfg['input_root']).rglob('*'):
        if p.is_dir() and canonical_generator(p.name):
            if any((p / s).is_dir() for s in ('train', 'training')):
                roots.add(p.parent)
    if len(roots) != 1:
        raise ValueError(f'Set dataset_root explicitly. Candidate roots: {sorted(map(str, roots))}')
    return roots.pop()


def scan_dataset(cfg, out):
    root = find_root(cfg)
    rows, errors = [], []
    extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    for g in sorted(root.iterdir()):
        gen = canonical_generator(g.name)
        if not g.is_dir() or gen is None:
            continue
        for split, names in [('train', ['train', 'training']), ('test', ['val', 'valid', 'validation', 'test'])]:
            available = [g / s for s in names if (g / s).is_dir()]
            if len(available) != 1:
                raise ValueError(f'{g}: expected exactly one {split} split, found {available}')
            labels = {0: [], 1: []}
            mapping = {'nature': 0, 'real': 0, '0_real': 0, 'ai': 1, 'fake': 1, '1_fake': 1}
            for folder in available[0].iterdir():
                if folder.is_dir() and folder.name.lower() in mapping:
                    labels[mapping[folder.name.lower()]].append(folder)
            if any(len(v) != 1 for v in labels.values()):
                raise ValueError(f'Unexpected label folders: {available[0]} {labels}')
            for label, folders in labels.items():
                for path in sorted(folders[0].rglob('*')):
                    if path.suffix.lower() not in extensions or not path.is_file():
                        continue
                    # Audit source-train only: other generators' train images are not used.
                    if split == 'train' and gen not in cfg.get('train_generators', [cfg['source_generator']]):
                        continue
                    try:
                        raw = path.read_bytes()
                        with Image.open(io.BytesIO(raw)) as im:
                            fmt, w, h = im.format, *im.size
                            qtable = im.quantization if hasattr(im, 'quantization') else None
                            rgb = im.convert('RGB')
                            pixel_hash = hashlib.sha256(f'{w}x{h}:'.encode() + rgb.tobytes()).hexdigest()
                        rel = path.relative_to(root).as_posix()
                        rows.append(dict(sample_id=hashlib.sha256(rel.encode()).hexdigest(),
                                         path=str(path), relative_path=rel, generator=gen, split=split,
                                         label=label, width=w, height=h, format=fmt,
                                         pixel_hash=pixel_hash, file_hash=hashlib.sha256(raw).hexdigest(),
                                         qtable_hash=digest(qtable) if qtable else '', file_bytes=len(raw)))
                        if cfg.get('scan_log_every') and len(rows) % cfg['scan_log_every'] == 0:
                            print(f'[scan] {len(rows):,} images; {gen}/{split}; invalid={len(errors)}', flush=True)
                    except Exception as e:
                        errors.append(dict(path=str(path), error=repr(e)))
    df = pd.DataFrame(rows)
    pd.DataFrame(errors).to_csv(out / 'invalid_images.csv', index=False)
    if df.empty:
        raise ValueError('No usable images')
    if (df.groupby('pixel_hash').label.nunique() > 1).any():
        raise ValueError('Identical decoded pixels have conflicting labels; inspect dataset')
    df.to_csv(out / 'image_audit.csv', index=False)
    df.groupby(['split', 'generator', 'label', 'format']).size().rename('count').to_csv(out / 'format_counts.csv')
    found = set(df.loc[df.split == 'test', 'generator'])
    expected = set(cfg.get('expected_generators') or [])
    if not cfg['smoke'] and expected and found != expected:
        raise ValueError(f'Test generators mismatch: missing={sorted(expected-found)}, '
                         f'unexpected={sorted(found-expected)}, found={sorted(found)}. '
                         'Inspect image_audit.csv and invalid_images.csv.')
    if cfg['strict_eight_generators'] and len(found) != 8:
        raise ValueError(f'Expected eight test generators; found {sorted(found)}')
    for source_gen in cfg.get('train_generators', [cfg['source_generator']]):
        source = df[(df.split == 'train') & (df.generator == source_gen)]
        if set(source.label) != {0, 1}:
            raise ValueError(f"Source {source_gen!r} needs real and fake training images. "
                             'yangsangtai/tiny-genimage excludes SD14; use source_generator=sd15.')
    return df


def balanced(frame, seed, limit=None):
    n = int(frame.label.value_counts().reindex([0, 1], fill_value=0).min())
    n = min(n, limit) if limit else n
    if n == 0:
        raise ValueError('Every cohort must contain real and fake')
    return pd.concat([frame[frame.label == y].sample(n=n, random_state=seed + y) for y in (0, 1)])


def make_splits(df, cfg, seed, out):
    source = df[(df.split == 'train') & (df.generator == cfg['source_generator'])].drop_duplicates('pixel_hash')
    if cfg['smoke']:
        source = balanced(source, seed, cfg['smoke_train_per_class'])
    source = balanced(source, seed)
    tr, va = train_test_split(source, test_size=cfg['validation_fraction'], stratify=source.label, random_state=seed)
    used = set(source.pixel_hash)
    tests = df[df.split == 'test'].copy()
    removed = tests[tests.pixel_hash.isin(used)]
    removed.to_csv(out / 'excluded_test_overlap.csv', index=False)
    tests = tests[~tests.pixel_hash.isin(used)]
    tests = tests.drop_duplicates(['generator', 'pixel_hash'])
    tests = pd.concat([balanced(g, seed, cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
                       for _, g in tests.groupby('generator')])
    # Label-blind order shared by all models and processing cases; never sort batches by label.
    def ordered(frame):
        frame = frame.copy()
        frame['_order'] = frame.sample_id.map(lambda s: digest([seed, s]))
        return frame.sort_values('_order').drop(columns='_order').reset_index(drop=True)
    tr, va, tests = ordered(tr), ordered(va), ordered(tests)
    for name, frame in [('train', tr), ('validation', va), ('test', tests)]:
        frame.to_csv(out / f'{name}_manifest.csv', index=False)
    dump(out / 'split_summary.json', dict(train=len(tr), validation=len(va), test=len(tests),
         test_overlap_removed=len(removed), test_repeated_real_pixels=int(tests[tests.label == 0].pixel_hash.duplicated().sum()),
         train_hash=digest(tr.sample_id.tolist()), validation_hash=digest(va.sample_id.tolist()),
         test_hash=digest(tests.sample_id.tolist()), source_generator=cfg['source_generator'],
         selection=f"source {cfg['source_generator']} train only; no test selection"))
    return tr, va, tests


def jpeg(image, quality, subsampling):
    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=int(quality), subsampling=subsampling,
               optimize=False, progressive=False)
    buf.seek(0)
    with Image.open(buf) as decoded:
        return decoded.convert('RGB')


def processing_cases(cfg):
    return ['native', 'size'] + [c for q in cfg['jpeg_eval_qualities'] for c in (f'jpeg{q}', f'size_jpeg{q}')]


def process(image, case, cfg):
    # Size stress-test includes square warp; explicitly NOT claimed to remove all size/history bias.
    if case.startswith('size'):
        image = image.resize((cfg['size_control'], cfg['size_control']), Image.Resampling.BICUBIC)
    if 'jpeg' in case:
        quality = int(case.split('jpeg')[-1])
        image = jpeg(image, quality, cfg['jpeg_subsampling'])
    return image


class Images(Dataset):
    def __init__(self, frame, cfg, seed, train=False, epoch=0, case='native'):
        self.rows = frame.to_dict('records')
        self.cfg, self.seed, self.train, self.epoch, self.case = cfg, seed, train, epoch, case

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r, c = self.rows[i], self.cfg
        with Image.open(r['path']) as im:
            image = im.convert('RGB')
        if self.train:
            rng = random.Random(int(digest([self.seed, self.epoch, r['sample_id']])[:16], 16))
            if rng.random() < c['horizontal_flip_probability']:
                image = ImageOps.mirror(image)
            if rng.random() < c['blur_probability']:
                image = image.filter(ImageFilter.GaussianBlur(rng.uniform(*c['blur_sigma'])))
            if rng.random() < c['jpeg_probability']:
                image = jpeg(image, rng.randint(*c['jpeg_train_quality']), c['jpeg_subsampling'])
        else:
            image = process(image, self.case, c)
        # Explicit standard OpenAI CLIP: shorter-side 224 bicubic, center crop, RGB mean/std.
        w, h = image.size
        image = image.resize((224 if w <= h else int(224 * w / h),
                              224 if h <= w else int(224 * h / w)), Image.Resampling.BICUBIC)
        w, h = image.size
        left, top = int(round((w - 224) / 2)), int(round((h - 224) / 2))
        image = image.crop((left, top, left + 224, top + 224))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.
        x = (x - torch.tensor([.48145466, .4578275, .40821073])[:, None, None]) / torch.tensor([.26862954, .26130258, .27577711])[:, None, None]
        return x, int(r['label']), i


def loader(frame, cfg, seed, train=False, epoch=0, case='native'):
    ds = Images(frame, cfg, seed, train, epoch, case)
    return DataLoader(ds, batch_size=cfg['train_batch'] if train else cfg['test_batch'],
                      shuffle=train, generator=torch.Generator().manual_seed(seed + epoch),
                      num_workers=cfg['num_workers'], pin_memory=torch.cuda.is_available(),
                      drop_last=train)


class SemanticSuppression(nn.Module):
    def __init__(self, a=2., b=1., maximum=1.):
        super().__init__()
        self.a, self.b = nn.Parameter(torch.tensor(float(a))), nn.Parameter(torch.tensor(float(b)))
        self.maximum = maximum

    def forward(self, features, basis):
        # FP32 projection/occupancy; do not detach occupancy (Eq.10 differentiated end-to-end).
        with torch.autocast(features.device.type, enabled=False):
            f = features.float()
            p = (f @ basis.float()) @ basis.float().T
            r = p.square().sum((1, 2)) / f.square().sum((1, 2)).clamp_min(1e-12)
            lam = self.maximum * torch.sigmoid(self.a * r + self.b)
            return (f - lam[:, None, None] * p).to(features.dtype)


def top_basis(matrix, k, driver=None):
    with torch.autocast(matrix.device.type, enabled=False):
        options = {'driver': driver} if matrix.is_cuda and driver else {}
        _, _, vh = torch.linalg.svd(matrix.float(), full_matrices=False, **options)
        if k > min(matrix.shape):
            raise ValueError('k exceeds available dimensions')
        return vh[:k].T.contiguous().detach()


class Detector(nn.Module):
    def __init__(self, vision, cfg, gsd):
        super().__init__()
        self.vision = vision
        self.cfg = cfg
        self.selected = tuple(range(len(vision.encoder.layers) - 4, len(vision.encoder.layers)))
        d = vision.config.hidden_size
        self.head = nn.Linear(d, 2)
        self.gsd = nn.ModuleDict({str(l): SemanticSuppression(cfg['a_init'], cfg['b_init'], cfg['lambda_max'])
                                  for l in self.selected}) if gsd else nn.ModuleDict()

    def forward(self, x, bases=None):
        h = self.vision.pre_layrnorm(self.vision.embeddings(x))
        for l, block in enumerate(self.vision.encoder.layers):
            if str(l) in self.gsd:
                h = self.gsd[str(l)](h, bases[l])
            def apply(t, layer=block):
                return layer(t, attention_mask=None, causal_attention_mask=None)[0]
            h = checkpoint(apply, h, use_reentrant=False) if self.training and self.cfg['gradient_checkpointing'] else apply(h)
        h = self.vision.post_layernorm(h)
        if self.cfg['pooling'] == 'patch_mean_post_ln':
            h = h[:, 1:].mean(1)
        elif self.cfg['pooling'] == 'all_mean_post_ln':
            h = h.mean(1)
        else:
            raise ValueError('Unsupported pooling')
        return self.head(h)


def amp(cfg, device):
    return torch.autocast(device.type, dtype=torch.float16 if cfg['precision'] == 'fp16' else torch.bfloat16,
                          enabled=device.type == 'cuda' and cfg['precision'] != 'fp32')


@torch.no_grad()
def estimate_bases(anchor, images, cfg, device, selected):
    # Anchor PRE-block states, matching Fig.3. Collect the ENTIRE logical batch on CPU.
    chunks = {l: [] for l in selected}
    for x in images.split(cfg['anchor_micro_batch']):
        with amp(cfg, device):
            h = anchor.pre_layrnorm(anchor.embeddings(x.to(device)))
            for l, layer in enumerate(anchor.encoder.layers):
                if l in chunks:
                    chunks[l].append(h[:, 1:].detach().cpu())
                h = layer(h, attention_mask=None, causal_attention_mask=None)[0]
    bases = {}
    for l in selected:
        m = torch.cat(chunks.pop(l), dim=0).flatten(0, 1).to(device=device, dtype=torch.float32)
        bases[l] = top_basis(m, cfg['k'], cfg['svd_driver'])
        del m
    return bases


def build_models(cfg, device, gsd):
    pretrained = CLIPVisionModel.from_pretrained(cfg['model_id'], local_files_only=cfg['local_files_only'], attn_implementation='eager')
    v = pretrained.vision_model
    assert (v.config.hidden_size, v.config.num_hidden_layers, v.config.patch_size, v.config.image_size) == (1024, 24, 14, 224), 'Expected CLIP ViT-L/14 224'
    model = Detector(v, cfg, gsd).to(device)
    anchor = copy.deepcopy(v).requires_grad_(False).eval().to(device) if gsd else None
    if anchor is not None and device.type == 'cuda' and cfg['precision'] != 'fp32':
        anchor.to(dtype=torch.float16 if cfg['precision'] == 'fp16' else torch.bfloat16)
    return model, anchor


@torch.no_grad()
def predict(model, anchor, frame, cfg, seed, device, case):
    model.eval()
    result = np.full(len(frame), np.nan)
    batches = np.full(len(frame), -1, dtype=np.int64)
    for batch_id, (images, _, indices) in enumerate(tqdm(loader(frame, cfg, seed, case=case), desc=case)):
        bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
        for start in range(0, len(images), cfg['micro_batch']):
            ix = indices[start:start + cfg['micro_batch']].numpy()
            with amp(cfg, device):
                logits = model(images[start:start + cfg['micro_batch']].to(device), bases)
            result[ix] = logits.float().softmax(-1)[:, 1].cpu().numpy()
            batches[ix] = batch_id
    if not np.isfinite(result).all():
        raise ValueError('Nonfinite or missing predictions')
    return result, batches


def metric(y, p, threshold):
    pred = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(n=len(y), auc=float(roc_auc_score(y, p)), ap=float(average_precision_score(y, p)),
                balanced_accuracy=float(balanced_accuracy_score(y, pred)), accuracy=float(np.mean(y == pred)),
                fake_recall=float(tp / (tp + fn)), real_recall=float(tn / (tn + fp)),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))


class EarlyStopping:
    """Track significant validation-loss improvements, separately from best checkpoint."""
    def __init__(self, enabled=True, patience=3, min_delta=0.001, state=None):
        if patience < 1 or min_delta < 0:
            raise ValueError('patience must be >=1; min_delta must be >=0')
        self.enabled, self.patience, self.min_delta = enabled, patience, min_delta
        self.state = dict(reference_loss=float('inf'), bad_epochs=0, stopped=False)
        if state is not None:
            self.state.update(state)

    def update(self, loss):
        if not math.isfinite(loss):
            raise FloatingPointError('Nonfinite validation loss')
        if loss < self.state['reference_loss'] - self.min_delta:
            self.state.update(reference_loss=loss, bad_epochs=0)
        else:
            self.state['bad_epochs'] += 1
        self.state['stopped'] = bool(self.enabled and self.state['bad_epochs'] >= self.patience)
        return self.state['stopped']


def train_model(model, anchor, train, val, cfg, seed, device, folder, signature):
    groups = [dict(params=model.vision.parameters(), lr=cfg['lr']),
              dict(params=model.head.parameters(), lr=cfg['head_lr'])]
    if len(model.gsd):
        groups.append(dict(params=model.gsd.parameters(), lr=cfg['modulation_lr']))
    optimizer = torch.optim.AdamW(groups, betas=tuple(cfg['betas']), eps=cfg['eps'], weight_decay=cfg['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and cfg['precision'] == 'fp16')
    start, best, history = 0, float('inf'), []
    stopper = EarlyStopping(cfg['early_stopping'], cfg['patience'], cfg['min_delta'])
    latest = folder / 'last.pt'
    if cfg['resume'] and latest.exists():
        ck = torch.load(latest, map_location='cpu', weights_only=False)
        if ck['signature'] != signature:
            raise ValueError('Resume protocol/config/manifest changed; use a fresh output_root')
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        start, best, history = ck['epoch'] + 1, ck['best'], ck['history']
        if 'early_stopping' in ck:
            stopper.state.update(ck['early_stopping'])
        del ck
    if len(train) < cfg['train_batch']:
        raise ValueError('Not enough source-training images for a full logical batch')
    for epoch in range(start, cfg['epochs']):
        if stopper.state['stopped']:
            print('Resuming an early-stopped run: restore best checkpoint and evaluate.')
            break
        model.train()
        running, count, skipped = 0., 0, 0
        for images, labels, _ in tqdm(loader(train, cfg, seed, True, epoch), desc=f'epoch {epoch+1}'):
            bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
            optimizer.zero_grad(set_to_none=True)
            for a in range(0, len(images), cfg['micro_batch']):
                x, y = images[a:a + cfg['micro_batch']].to(device), labels[a:a + cfg['micro_batch']].to(device)
                with amp(cfg, device):
                    logits = model(x, bases)
                    loss = F.cross_entropy(logits.float(), y, reduction='sum') / len(images)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite training loss')
                scaler.scale(loss).backward()
                running += float(loss.detach()) * len(images)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += int(scaler.get_scale() < old_scale)
            count += len(images)
        p, _ = predict(model, anchor, val, cfg, seed, device, 'native')
        vloss = float(F.binary_cross_entropy(torch.from_numpy(p).double().clamp(1e-7, 1-1e-7),
                                             torch.tensor(val.label.to_numpy(), dtype=torch.float64)))
        entry = dict(epoch=epoch+1, train_loss=running/count, val_loss=vloss, amp_skipped_steps=skipped,
                     **metric(val.label.to_numpy(), p, cfg['threshold']))
        entry['modulation'] = {l: dict(a=float(g.a), b=float(g.b)) for l, g in model.gsd.items()}
        should_stop = stopper.update(vloss)
        entry['early_stopping'] = dict(stopper.state)
        history.append(entry)
        print(entry)
        if vloss < best:
            best = vloss
            torch.save(dict(model=model.state_dict(), epoch=epoch, signature=signature, config=cfg), folder / 'best.pt')
        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                        epoch=epoch, best=best, history=history, signature=signature,
                        early_stopping=dict(stopper.state)), latest)
        dump(folder / 'history.json', history)
        dump(folder / 'training_status.json', dict(epoch=epoch+1, max_epochs=cfg['epochs'],
             early_stopping=dict(stopper.state), patience=cfg['patience'], min_delta=cfg['min_delta']))
        if should_stop:
            print(f"Early stopping at epoch {epoch+1}: no significant val-loss improvement for {cfg['patience']} epochs.")
            break
    ck = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    dump(folder / 'selected_checkpoint.json', dict(epoch=ck['epoch']+1, criterion='minimum source-validation cross entropy', signature=signature))


def summarize(predictions, cfg, out):
    metrics, shifts = [], []
    for (seed, model, case), frame in predictions.groupby(['seed', 'model', 'case']):
        for gen, g in [('pooled', frame)] + list(frame.groupby('generator')):
            metrics.append(dict(seed=seed, model=model, case=case, generator=gen,
                                seen=(gen == cfg['source_generator']), **metric(g.label.to_numpy(), g.prob_fake.to_numpy(), cfg['threshold'])))
    metric_df = pd.DataFrame(metrics)
    metric_df.to_csv(out / 'metrics.csv', index=False)
    per_gen = metric_df[metric_df.generator != 'pooled']
    for protocol, g in [('all', per_gen), ('unseen', per_gen[per_gen.generator != cfg['source_generator']])]:
        g.groupby(['seed', 'model', 'case'])[['auc', 'balanced_accuracy', 'fake_recall', 'real_recall', 'accuracy']].mean().to_csv(out / f'macro_{protocol}.csv')
    for (seed, model), group in predictions.groupby(['seed', 'model']):
        base = group[group.case == 'native'].set_index('sample_id')
        for case, changed in group[group.case != 'native'].groupby('case'):
            changed = changed.set_index('sample_id')
            if set(changed.index) != set(base.index):
                raise ValueError('Paired cohorts differ')
            changed = changed.loc[base.index]
            if not np.array_equal(base.batch_id, changed.batch_id):
                raise ValueError('GSD logical batch membership changed')
            delta = changed.prob_fake - base.prob_fake
            paired = base[['generator', 'label']].copy()
            paired['delta'] = delta
            paired['abs_delta'] = delta.abs()
            paired['flip'] = (changed.prob_fake >= cfg['threshold']) != (base.prob_fake >= cfg['threshold'])
            paired['native_prob'] = base.prob_fake
            paired['changed_prob'] = changed.prob_fake
            paired.reset_index().to_csv(out / f'paired_seed{seed}_{model}_{case}.csv', index=False)
            for (gen, label), g in paired.groupby(['generator', 'label']):
                shifts.append(dict(seed=seed, model=model, case=case, generator=gen, label=label,
                                   mean_delta=g.delta.mean(), mean_abs_delta=g.abs_delta.mean(), flip_rate=g.flip.mean(), n=len(g)))
    pd.DataFrame(shifts).to_csv(out / 'paired_summary.csv', index=False)
    comparison = per_gen.pivot(index=['seed', 'case', 'generator'], columns='model', values=['auc', 'balanced_accuracy', 'fake_recall'])
    comparison.to_csv(out / 'model_comparison.csv')
    if {'clip_ft', 'clip_gsd'} <= set(predictions.model):
        for name in ('auc', 'balanced_accuracy', 'fake_recall'):
            comparison[(name, 'gsd_minus_ft')] = comparison[(name, 'clip_gsd')] - comparison[(name, 'clip_ft')]
        comparison.to_csv(out / 'model_comparison.csv')
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    plot = per_gen[per_gen.generator != cfg['source_generator']].groupby(['model', 'case']).balanced_accuracy.mean().unstack(0)
    plot.plot.bar(ax=ax)
    ax.set_ylabel('Macro balanced accuracy (unseen generators)')
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(out / 'unseen_balanced_accuracy.png', dpi=160)
    plt.close(fig)
    return metric_df


def run(config=None):
    cfg = {**DEFAULT_CONFIG, **(config or {})}
    if cfg['smoke']:
        cfg.update(epochs=1, train_batch=8, test_batch=8, k=4, strict_eight_generators=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type != 'cuda':
        raise RuntimeError('Use Kaggle GPU; run local unit tests for CPU validation')
    if cfg['precision'] == 'bf16' and not torch.cuda.is_bf16_supported():
        raise ValueError('This GPU does not support BF16; use fp16 on T4/P100')
    if min(cfg['epochs'], cfg['micro_batch'], cfg['anchor_micro_batch'], cfg['train_batch'], cfg['test_batch']) < 1:
        raise ValueError('Batch sizes and epoch count must be positive')
    out = Path(cfg['output_root'])
    out.mkdir(parents=True, exist_ok=True)
    dump(out / 'config.json', cfg)
    import transformers
    dump(out / 'environment.json', dict(torch=torch.__version__, transformers=transformers.__version__,
         pillow=PIL.__version__, python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
         use_devices=1, paper='https://arxiv.org/html/2603.09242v2', torch_threads=torch.get_num_threads()))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    df = scan_dataset(cfg, out)
    predictions = []
    for seed in cfg['seeds']:
        split_out = out / f'seed{seed}'
        split_out.mkdir(exist_ok=True)
        train, val, tests = make_splits(df, cfg, seed, split_out)
        cfg_hash = digest({**cfg, 'train_files': train.file_hash.tolist(), 'val_files': val.file_hash.tolist(),
                           'test_files': tests.file_hash.tolist()})
        for model_name in cfg['models']:
            if model_name not in ('clip_ft', 'clip_gsd'):
                raise ValueError(model_name)
            seed_all(seed)  # Same backbone AND randomly initialized classifier across the two models.
            folder = split_out / model_name
            folder.mkdir(exist_ok=True)
            model, anchor = build_models(cfg, device, model_name == 'clip_gsd')
            dump(folder / 'architecture.json', dict(backbone=str(model.vision.config),
                 trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                 gsd_pre_block_indices=list(model.selected) if anchor else [],
                 initialization_head_sha=digest(model.head.weight.detach().cpu().tolist()),
                 pooling=cfg['pooling'], batch_basis='entire logical batch, independent of microbatch'))
            train_model(model, anchor, train, val, cfg, seed, device, folder, digest([cfg_hash, model_name]))
            for case in processing_cases(cfg):
                p, batches = predict(model, anchor, tests, cfg, seed, device, case)
                frame = tests[['sample_id', 'generator', 'label', 'pixel_hash']].copy()
                frame = frame.assign(seed=seed, model=model_name, case=case, prob_fake=p, batch_id=batches)
                frame.to_csv(folder / f'predictions_{case}.csv', index=False)
                predictions.append(frame)
            del model, anchor
            gc.collect()
            torch.cuda.empty_cache()
    predictions = pd.concat(predictions, ignore_index=True)
    predictions.to_csv(out / 'predictions.csv', index=False)
    metrics = summarize(predictions, cfg, out)
    dump(out / 'completion.json', dict(status='complete', smoke=cfg['smoke'], config_hash=digest(cfg)))
    print(metrics[metrics.generator == 'pooled'].to_string(index=False))
    return metrics


"""Native-resolution, raw-only All7 training. Also appended to the base runtime in the notebook."""
if 'DEFAULT_CONFIG' not in globals():
    from pathlib import Path
    _base_file = Path(__file__).with_name('runtime.py')
    exec(compile(_base_file.read_text(encoding='utf-8'), str(_base_file), 'exec'), globals())

import time

GENS7 = ['sd15', 'adm', 'biggan', 'glide', 'midjourney', 'wukong', 'vqdm']
DEFAULT_CONFIG = dict(DEFAULT_CONFIG, output_root='/kaggle/working/gsd_raw_all7_native_v1',
    source_generator='all7', train_generators=GENS7, expected_generators=GENS7,
    train_batch=8, test_batch=8, micro_batch=1, anchor_micro_batch=1, num_workers=0,
    jpeg_probability=0., blur_probability=0., horizontal_flip_probability=0.,
    input_mode='native_rgb_edge_pad14', basis_method='gram_eigh', attention='sdpa',
    eval_cases=['native'], scan_log_every=500)

_base_scan = scan_dataset

def scan_dataset(cfg, out):
    print('[scan] Reading all seven train/val folders; RGB decoding and hashing. No resize.', flush=True)
    start = time.monotonic()
    frame = _base_scan(cfg, out)
    frame.groupby(['split','generator','label']).size().rename('n').to_csv(out/'dataset_counts.csv')
    print(f'[scan] {len(frame):,} images audited in {time.monotonic()-start:.1f}s', flush=True)
    return frame


def make_splits(df, cfg, seed, out):
    source = df[(df.split=='train') & df.generator.isin(cfg['train_generators'])].copy()
    # Assign each decoded-pixel group ONCE, even when it occurs in several generators.
    unique = source.drop_duplicates('pixel_hash')
    strata = unique.generator + ':' + unique.label.astype(str)
    a, b = train_test_split(unique, test_size=cfg['validation_fraction'],
                          stratify=strata, random_state=seed)
    def select(frame, limit):
        groups = [g.drop_duplicates('pixel_hash') for _,g in frame.groupby('generator',sort=True)]
        if set(frame.generator)!=set(cfg['train_generators']):
            raise ValueError('Every configured train generator must have a nonempty cohort')
        n = min(int(g.label.value_counts().reindex([0,1],fill_value=0).min()) for g in groups)
        if limit is not None:
            n = min(n, limit)
        if n < 1:
            raise ValueError('A generator lacks real/fake images after hash-group splitting')
        return pd.concat([balanced(g, seed, n) for g in groups],ignore_index=True)
    tr = select(source[source.pixel_hash.isin(a.pixel_hash)], cfg['smoke_train_per_class'] if cfg['smoke'] else None)
    va = select(source[source.pixel_hash.isin(b.pixel_hash)], cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
    tests = df[df.split=='test'].copy()
    removed = tests[tests.pixel_hash.isin(source.pixel_hash)]
    removed.to_csv(out/'excluded_test_overlap.csv',index=False)
    tests = tests[~tests.pixel_hash.isin(source.pixel_hash)].drop_duplicates(['generator','pixel_hash'])
    tests = pd.concat([balanced(g,seed,cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
                       for _,g in tests.groupby('generator',sort=True)],ignore_index=True)
    def ordered(f):
        return f.assign(_order=f.sample_id.map(lambda s:digest([seed,s]))).sort_values('_order').drop(columns='_order').reset_index(drop=True)
    tr,va,tests=map(ordered,(tr,va,tests))
    assert not set(tr.pixel_hash)&set(va.pixel_hash)
    assert not (set(tr.pixel_hash)|set(va.pixel_hash))&set(tests.pixel_hash)
    for name,frame in [('train',tr),('validation',va),('test',tests)]:
        frame.to_csv(out/f'{name}_manifest.csv',index=False)
        frame.groupby(['generator','label']).size().rename('n').to_csv(out/f'{name}_counts.csv')
    dump(out/'split_summary.json',dict(train=len(tr),validation=len(va),test=len(tests),
        test_overlap_removed=len(removed),selection='All7 raw; unique pixel groups split before per-generator/class balancing',
        train_hash=digest(tr.sample_id.tolist()),validation_hash=digest(va.sample_id.tolist()),test_hash=digest(tests.sample_id.tolist())))
    print(f'[split] train={len(tr):,}, validation={len(va):,}, test={len(tests):,}; all 7 seen',flush=True)
    return tr,va,tests


def processing_cases(cfg):
    return cfg['eval_cases']


def input_geometry(width, height, cfg):
    """Source-pixel crop coordinates and exact model dimensions, without resampling."""
    cap = cfg['crop_max_side'] if cfg['input_mode']=='center_crop_max_edge_pad14' else None
    cw = min(width, cap) if cap else width
    ch = min(height, cap) if cap else height
    left, top = (width-cw)//2, (height-ch)//2
    pw, ph = ((cw+13)//14)*14, ((ch+13)//14)*14
    return dict(crop_left=left,crop_top=top,crop_width=cw,crop_height=ch,
                input_width=pw,input_height=ph,grid_w=pw//14,grid_h=ph//14,
                tokens=(pw//14)*(ph//14)+1)


def protocol_id(cfg):
    if cfg['input_mode']=='center_crop_max_edge_pad14':
        return f"raw_all7_center_crop{cfg['crop_max_side']}_v1"
    return 'raw_all7_native_v1'


class Images(Dataset):
    def __init__(self,frame,cfg,seed,train=False,epoch=0,case='native'):
        self.rows=frame.to_dict('records'); self.cfg=cfg; self.case=case; self.train=train
        if train and case!='native':
            raise ValueError('Training must use native raw images')

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,i):
        row=self.rows[i]
        with Image.open(row['path']) as im:
            image=im.convert('RGB')
        if not self.train:
            if self.case.startswith('fake_jpeg'):
                if int(row['label'])==1:
                    image=jpeg(image,int(self.case[len('fake_jpeg'):]),self.cfg['jpeg_subsampling'])
            elif self.case!='native':
                image=process(image,self.case,self.cfg)
        geom=input_geometry(*image.size,self.cfg)
        image=image.crop((geom['crop_left'],geom['crop_top'],
                          geom['crop_left']+geom['crop_width'],geom['crop_top']+geom['crop_height']))
        x=torch.from_numpy(np.asarray(image,dtype=np.float32).copy()).permute(2,0,1)/255.
        # Preserve pixels inside the selected crop. No resize or upscaling of smaller images.
        h,w=x.shape[-2:]
        x=F.pad(x.unsqueeze(0),(0,(-w)%14,0,(-h)%14),mode='replicate').squeeze(0)
        mean=x.new_tensor([.48145466,.4578275,.40821073])[:,None,None]
        std=x.new_tensor([.26862954,.26130258,.27577711])[:,None,None]
        return (x-mean)/std,int(row['label']),i


def native_collate(items):
    x,y,i=zip(*items)
    return list(x),torch.tensor(y,dtype=torch.long),torch.tensor(i,dtype=torch.long)


def loader(frame,cfg,seed,train=False,epoch=0,case='native'):
    return DataLoader(Images(frame,cfg,seed,train,epoch,case),
        batch_size=cfg['train_batch'] if train else cfg['test_batch'], shuffle=train,
        generator=torch.Generator().manual_seed(seed+epoch),num_workers=cfg['num_workers'],
        collate_fn=native_collate,pin_memory=torch.cuda.is_available(),drop_last=False)


def image_chunks(images,limit):
    # Keep membership and order of the mixed logical batch, regardless of shape or label.
    start=0
    while start<len(images):
        end=start+1
        while end<min(start+limit,len(images)) and images[end].shape==images[start].shape:
            end+=1
        yield start,torch.stack(images[start:end])
        start=end


class NativeDetector(Detector):
    def forward(self,x,bases=None):
        h=self.vision.pre_layrnorm(self.vision.embeddings(x,interpolate_pos_encoding=True))
        for l,block in enumerate(self.vision.encoder.layers):
            if str(l) in self.gsd:
                h=self.gsd[str(l)](h,bases[l])
            def apply(t,layer=block):
                return layer(t,attention_mask=None,causal_attention_mask=None)[0]
            h=checkpoint(apply,h,use_reentrant=False) if self.training and self.cfg['gradient_checkpointing'] else apply(h)
        return self.head(self.vision.post_layernorm(h)[:,1:].mean(1))


@torch.no_grad()
def estimate_bases(anchor,images,cfg,device,selected):
    # M.T @ M has the same principal right-singular subspace as uncentered M.
    # Stream ALL tokens: no token sampling and no BN x d concatenation on GPU.
    grams={l:torch.zeros(anchor.config.hidden_size,anchor.config.hidden_size,device=device,dtype=torch.float32) for l in selected}
    tokens=0
    for _,x in image_chunks(images,cfg['anchor_micro_batch']):
        with amp(cfg,device):
            h=anchor.pre_layrnorm(anchor.embeddings(x.to(device),interpolate_pos_encoding=True))
            tokens+=h.shape[0]*(h.shape[1]-1)
            for l,layer in enumerate(anchor.encoder.layers):
                if l in grams:
                    with torch.autocast(device.type,enabled=False):
                        m=h[:,1:].flatten(0,1).float()
                        grams[l].add_(m.T@m)
                h=layer(h,attention_mask=None,causal_attention_mask=None)[0]
    if cfg['k']>min(tokens,anchor.config.hidden_size):
        raise ValueError(f"k={cfg['k']} exceeds token/feature dimensions of this logical batch ({tokens})")
    bases={}
    with torch.autocast(device.type,enabled=False):
        for l,g in grams.items():
            _,v=torch.linalg.eigh((g+g.T)*(0.5/tokens))
            bases[l]=v[:,-cfg['k']:].contiguous().detach()
    return bases


def build_models(cfg,device,gsd):
    pretrained=CLIPVisionModel.from_pretrained(cfg['model_id'],local_files_only=cfg['local_files_only'],attn_implementation=cfg['attention'])
    v=pretrained.vision_model
    if (v.config.hidden_size,v.config.patch_size)!=(1024,14):
        raise ValueError('Native All7 protocol requires CLIP ViT-L/14')
    model=NativeDetector(v,cfg,gsd).to(device)
    anchor=copy.deepcopy(v).requires_grad_(False).eval().to(device) if gsd else None
    if anchor is not None and device.type=='cuda' and cfg['precision']!='fp32':
        anchor.to(dtype=torch.float16 if cfg['precision']=='fp16' else torch.bfloat16)
    return model,anchor


@torch.no_grad()
def predict(model,anchor,frame,cfg,seed,device,case):
    model.eval(); result=np.full(len(frame),np.nan); batches=np.full(len(frame),-1)
    for bid,(images,_,indices) in enumerate(loader(frame,cfg,seed,case=case)):
        bases=estimate_bases(anchor,images,cfg,device,model.selected) if anchor is not None else None
        for start,x in image_chunks(images,cfg['micro_batch']):
            with amp(cfg,device):
                logits=model(x.to(device),bases)
            ix=indices[start:start+len(x)].numpy()
            result[ix]=logits.float().softmax(-1)[:,1].cpu().numpy(); batches[ix]=bid
        if bid%25==0:
            print(f'[{case}] batch {bid+1}/{math.ceil(len(frame)/cfg["test_batch"])}',flush=True)
    if not np.isfinite(result).all():
        raise ValueError('Missing/nonfinite prediction')
    return result,batches


def save_checkpoint(path,state):
    temp=path.with_suffix('.tmp')
    torch.save(state,temp)
    temp.replace(path)


def train_model(model,anchor,train,val,cfg,seed,device,folder,signature):
    groups=[dict(params=model.vision.parameters(),lr=cfg['lr']),dict(params=model.head.parameters(),lr=cfg['head_lr'])]
    if len(model.gsd):
        groups.append(dict(params=model.gsd.parameters(),lr=cfg['modulation_lr']))
    optimizer=torch.optim.AdamW(groups,betas=tuple(cfg['betas']),eps=cfg['eps'],weight_decay=cfg['weight_decay'])
    scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda' and cfg['precision']=='fp16')
    stopper=EarlyStopping(cfg['early_stopping'],cfg['patience'],cfg['min_delta'])
    latest=folder/'last.pt'; start=0; best=float('inf'); history=[]
    if cfg['resume'] and latest.exists():
        ck=torch.load(latest,map_location='cpu',weights_only=False)
        if ck['signature']!=signature:
            raise ValueError('Resume config/manifests changed; use a new output_root')
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); scaler.load_state_dict(ck['scaler'])
        stopper.state.update(ck['early_stopping']); start=ck['epoch']+1; best=ck['best']; history=ck['history']
        del ck
        print(f'[resume] next epoch {start+1}, stopped={stopper.state["stopped"]}',flush=True)
    for epoch in range(start,cfg['epochs']):
        if stopper.state['stopped']:
            break
        model.train(); running=0.; count=0; skipped=0; tick=time.monotonic()
        for bid,(images,labels,_) in enumerate(loader(train,cfg,seed,True,epoch)):
            bases=estimate_bases(anchor,images,cfg,device,model.selected) if anchor is not None else None
            optimizer.zero_grad(set_to_none=True)
            for a,x in image_chunks(images,cfg['micro_batch']):
                with amp(cfg,device):
                    logits=model(x.to(device),bases)
                    loss=F.cross_entropy(logits.float(),labels[a:a+len(x)].to(device),reduction='sum')/len(images)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite loss')
                scaler.scale(loss).backward(); running+=float(loss.detach())*len(images)
            old=scaler.get_scale(); scaler.step(optimizer); scaler.update(); skipped+=int(scaler.get_scale()<old); count+=len(images)
            if bid%10==0:
                print(f'[epoch {epoch+1}] batch {bid+1}/{math.ceil(len(train)/cfg["train_batch"])}; loss={running/count:.5f}',flush=True)
        p,_=predict(model,anchor,val,cfg,seed,device,'native')
        vloss=float(F.binary_cross_entropy(torch.from_numpy(p).double().clamp(1e-7,1-1e-7),torch.tensor(val.label.to_numpy(),dtype=torch.float64)))
        stop=stopper.update(vloss)
        entry=dict(epoch=epoch+1,train_loss=running/count,val_loss=vloss,seconds=time.monotonic()-tick,
            amp_skipped_steps=skipped,train_images=count,**metric(val.label.to_numpy(),p,cfg['threshold']),
            modulation={l:dict(a=float(g.a.detach()),b=float(g.b.detach())) for l,g in model.gsd.items()},early_stopping=dict(stopper.state))
        history.append(entry); print(entry,flush=True)
        if vloss<best:
            best=vloss; save_checkpoint(folder/'best.pt',dict(model=model.state_dict(),epoch=epoch,signature=signature,config=cfg))
        save_checkpoint(latest,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            epoch=epoch,best=best,history=history,signature=signature,config=cfg,early_stopping=dict(stopper.state)))
        dump(folder/'history.json',history)
        dump(folder/'training_status.json',dict(epoch=epoch+1,max_epochs=cfg['epochs'],early_stopping=dict(stopper.state)))
        if stop:
            print('[early stopping] Restore best checkpoint for evaluation.',flush=True)
            break
    ck=torch.load(folder/'best.pt',map_location='cpu',weights_only=False); model.load_state_dict(ck['model'])
    dump(folder/'selected_checkpoint.json',dict(epoch=ck['epoch']+1,criterion='minimum All7 raw validation cross entropy',signature=signature))


def summarize(predictions,cfg,out):
    rows=[]
    for (seed,model,case),f in predictions.groupby(['seed','model','case']):
        for gen,g in [('pooled',f),*f.groupby('generator')]:
            rows.append(dict(seed=seed,model=model,case=case,generator=gen,
                seen=gen in cfg['train_generators'],**metric(g.label.to_numpy(),g.prob_fake.to_numpy(),cfg['threshold'])))
    result=pd.DataFrame(rows); result.to_csv(out/'metrics.csv',index=False)
    result[result.generator!='pooled'].groupby(['seed','model','case'])[['auc','balanced_accuracy','fake_recall','real_recall','accuracy']].mean().to_csv(out/'macro_all.csv')
    return result


def validate_raw_config(cfg):
    if set(cfg['train_generators'])!=set(GENS7) or set(cfg['expected_generators'])!=set(GENS7):
        raise ValueError('This protocol requires exactly all seven Tiny generators')
    if any(cfg[k]!=0 for k in ('jpeg_probability','blur_probability','horizontal_flip_probability')):
        raise ValueError('Raw-only training forbids JPEG/blur/flip augmentation')
    if cfg['input_mode'] not in ('native_rgb_edge_pad14','center_crop_max_edge_pad14') or cfg['basis_method']!='gram_eigh':
        raise ValueError('Unsupported native-input protocol')
    if cfg['input_mode']=='center_crop_max_edge_pad14':
        cap=cfg.get('crop_max_side')
        if isinstance(cap,bool) or not isinstance(cap,int) or cap<14 or cap%14:
            raise ValueError('crop_max_side must be a positive integer multiple of 14')
    if cfg['attention']!='sdpa' or cfg['pooling']!='patch_mean_post_ln':
        raise ValueError('Native protocol requires SDPA and patch_mean_post_ln')
    if min(cfg[k] for k in ('train_batch','test_batch','micro_batch','anchor_micro_batch','epochs','k'))<1:
        raise ValueError('Positive batches, epochs and k required')
    if cfg['precision'] not in ('fp16','bf16','fp32'):
        raise ValueError('precision must be fp16, bf16 or fp32')
    if not cfg['eval_cases'] or 'native' not in cfg['eval_cases']:
        raise ValueError('Evaluation must include native')
    for case in cfg['eval_cases']:
        if case not in ('native','size') and not any(case==prefix+str(q) for prefix in ('jpeg','fake_jpeg','size_jpeg') for q in range(1,101)):
            raise ValueError(f'Unknown evaluation case: {case}')


def run(config=None):
    cfg={**DEFAULT_CONFIG,**(config or {})}; validate_raw_config(cfg)
    if cfg['smoke']:
        cfg.update(epochs=1,train_batch=2,test_batch=2,k=4)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type!='cuda':
        raise RuntimeError('Use Kaggle GPU; CPU unit tests cover native input separately')
    if cfg['precision']=='bf16' and not torch.cuda.is_bf16_supported():
        raise ValueError('GPU does not support bf16; use fp16 on T4')
    out=Path(cfg['output_root']); out.mkdir(parents=True,exist_ok=True)
    if (out/'config.json').exists():
        previous=json.loads((out/'config.json').read_text(encoding='utf-8'))
        if previous!=cfg:
            raise ValueError('Existing output config differs; choose a new output_root before running')
    dump(out/'config.json',cfg)
    dump(out/'environment.json',dict(torch=torch.__version__,pillow=PIL.__version__,python=platform.python_version(),
        gpu=torch.cuda.get_device_name(0),use_devices=1,protocol=protocol_id(cfg)))
    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
    df=scan_dataset(cfg,out)
    census=df[['sample_id','split','generator','label','width','height']].copy()
    geometry=pd.DataFrame([input_geometry(int(w),int(h),cfg) for w,h in zip(census.width,census.height)])
    census=pd.concat([census.reset_index(drop=True),geometry],axis=1)
    census.to_csv(out/'token_census.csv',index=False)
    print(f'[{protocol_id(cfg)}] tokens min/median/max: {census.tokens.min()}/{census.tokens.median()}/{census.tokens.max()}. No pixel resampling.',flush=True)
    results=[]
    for seed in cfg['seeds']:
        split_out=out/f'seed{seed}'; split_out.mkdir(exist_ok=True)
        train,val,tests=make_splits(df,cfg,seed,split_out)
        signature=digest(dict(config=cfg,train=train.file_hash.tolist(),val=val.file_hash.tolist(),test=tests.file_hash.tolist(),protocol=protocol_id(cfg)))
        for name in cfg['models']:
            if name not in ('clip_ft','clip_gsd'):
                raise ValueError(name)
            seed_all(seed); folder=split_out/name; folder.mkdir(exist_ok=True)
            print(f'[model] Loading {name}; native position interpolation in BOTH branches.',flush=True)
            model,anchor=build_models(cfg,device,name=='clip_gsd')
            dump(folder/'architecture.json',dict(backbone=str(model.vision.config),input_mode=cfg['input_mode'],
                basis=cfg['basis_method'],attention=cfg['attention'],all_generators_seen=True,patch_size=14,
                protocol=protocol_id(cfg),crop_max_side=cfg.get('crop_max_side'),
                padding='right/bottom edge replication to multiple of 14; no resize',
                trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)))
            try:
                train_model(model,anchor,train,val,cfg,seed,device,folder,signature)
                for case in processing_cases(cfg):
                    p,batches=predict(model,anchor,tests,cfg,seed,device,case)
                    f=tests[['sample_id','generator','label','pixel_hash','width','height']].assign(seed=seed,model=name,case=case,prob_fake=p,batch_id=batches)
                    f.to_csv(folder/f'predictions_{case}.csv',index=False); results.append(f)
            except torch.cuda.OutOfMemoryError as exc:
                raise RuntimeError('CUDA OOM under the configured input protocol. No automatic resize, skip or batch reduction. Use more VRAM or explicitly start a smaller-input/batch protocol; last.pt retains completed epochs.') from exc
            del model,anchor; gc.collect(); torch.cuda.empty_cache()
    predictions=pd.concat(results,ignore_index=True); predictions.to_csv(out/'predictions.csv',index=False)
    metrics=summarize(predictions,cfg,out)
    dump(out/'completion.json',dict(status='complete',smoke=cfg['smoke'],protocol=protocol_id(cfg),config_hash=digest(cfg)))
    print(metrics[metrics.generator=='pooled'].to_string(index=False),flush=True)
    return metrics


"""Trusted audit-manifest reuse. Loaded after runtime.py; never decode/hash image content.

The caller deliberately trusts the previous image audit and the same dataset version.
Filesystem stat checks cannot detect content changes that keep the same byte length.
"""
from pathlib import Path, PurePosixPath
import re

MANIFEST_PARTITIONS = ('train', 'validation', 'test')
MANIFEST_COLUMNS = {
    'sample_id', 'relative_path', 'generator', 'split', 'label', 'width', 'height',
    'file_hash', 'pixel_hash', 'file_bytes',
}


def find_manifest_root(cfg):
    def complete(root):
        return all((root / f'seed{seed}' / f'{part}_manifest.csv').is_file()
                   for seed in cfg['seeds'] for part in MANIFEST_PARTITIONS)

    if cfg['manifest_root']:
        root = Path(cfg['manifest_root']).resolve()
        if re.fullmatch(r'seed\d+', root.name):
            root = root.parent
        if not complete(root):
            raise FileNotFoundError(f'Missing saved train/validation/test manifests under {root}/seed<seed>. '
                                    'Attach the previous All7 output; no full-scan fallback is enabled.')
        return root
    search = Path(cfg['input_root'])
    roots = sorted({p.parent.parent.resolve() for p in search.rglob('train_manifest.csv')
                    if p.parent.name == f'seed{cfg["seeds"][0]}' and complete(p.parent.parent)})
    if len(roots) != 1:
        raise ValueError(f'Set manifest_root explicitly. Complete All7-manifest candidates: {[str(p) for p in roots]}. '
                         'Need seed42/train_manifest.csv, validation_manifest.csv and test_manifest.csv; '
                         'weights alone are not enough. No image scan was started.')
    return roots[0]


def validate_manifest_frame(frame, cfg, partition):
    missing = MANIFEST_COLUMNS - set(frame)
    if missing or frame.empty:
        raise ValueError(f'Invalid {partition} manifest: missing columns={sorted(missing)} or empty cohort')
    frame = frame.copy()
    if frame[list(MANIFEST_COLUMNS)].isna().any().any():
        raise ValueError(f'Null required metadata in {partition} manifest')
    if frame.sample_id.duplicated().any() or frame.relative_path.duplicated().any():
        raise ValueError(f'Duplicate IDs/paths in {partition} manifest')
    for column in ('sample_id', 'file_hash', 'pixel_hash'):
        if not frame[column].astype(str).str.fullmatch(r'[0-9a-f]{64}').all():
            raise ValueError(f'Invalid saved {column} in {partition} manifest')
    for column in ('label', 'width', 'height', 'file_bytes'):
        values = pd.to_numeric(frame[column], errors='coerce')
        if not np.isfinite(values).all() or (values != np.floor(values)).any():
            raise ValueError(f'Invalid integer {column} in {partition} manifest')
        frame[column] = values.astype('int64')
    if not frame.label.isin([0, 1]).all() or (frame[['width', 'height', 'file_bytes']] <= 0).any().any():
        raise ValueError(f'Invalid labels/dimensions/byte lengths in {partition} manifest')
    source_split = 'test' if partition == 'test' else 'train'
    if set(frame.split) != {source_split}:
        raise ValueError(f'{partition} manifest must originate only from source {source_split} folders')
    counts = frame.groupby(['generator', 'label']).size()
    expected = {(g, y) for g in cfg['train_generators'] for y in (0, 1)}
    if set(counts.index) != expected or counts.nunique() != 1:
        raise ValueError(f'{partition} manifest must contain balanced real/fake cohorts for all seven generators')
    if frame.duplicated(['generator', 'pixel_hash']).any():
        raise ValueError(f'Duplicate pixels within a generator in {partition} manifest')
    label_names = {'nature': 0, 'real': 0, '0_real': 0, 'ai': 1, 'fake': 1, '1_fake': 1}
    split_names = {'train': {'train', 'training'}, 'test': {'val', 'valid', 'validation', 'test'}}
    for row in frame.itertuples():
        rel = str(row.relative_path)
        parts = PurePosixPath(rel).parts
        if (not parts or len(parts) < 4 or PurePosixPath(rel).is_absolute() or
                '\\' in rel or ':' in rel or '..' in parts or rel != PurePosixPath(rel).as_posix()):
            raise ValueError(f'Unsafe relative image path: {rel}')
        if (canonical_generator(parts[0]) != row.generator or parts[1] not in split_names[source_split] or
                label_names.get(parts[2].lower()) != row.label):
            raise ValueError(f'Path/generator/split/label mismatch: {rel}')
        if row.sample_id != hashlib.sha256(rel.encode()).hexdigest():
            raise ValueError(f'Saved sample_id does not match relative path: {rel}')
    return frame


def read_manifest_cohorts(root, cfg, seed):
    folder = root / f'seed{seed}'
    summary_path = folder / 'split_summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else None
    frames, sources = {}, {}
    for partition in MANIFEST_PARTITIONS:
        path = folder / f'{partition}_manifest.csv'
        data = path.read_bytes()  # Hash only the small CSV, NOT any source image.
        frame = pd.read_csv(io.BytesIO(data), dtype={k: str for k in
            ('sample_id', 'relative_path', 'generator', 'split', 'file_hash', 'pixel_hash')})
        frame = validate_manifest_frame(frame, cfg, partition)
        if summary is not None:
            if (summary.get(partition) != len(frame) or
                    summary.get(partition + '_hash') != digest(frame.sample_id.tolist())):
                raise ValueError(f'{path}: ordered cohort does not match saved split_summary.json')
        frames[partition] = frame
        sources[partition] = dict(path=str(path), csv_sha256=hashlib.sha256(data).hexdigest(),
                                  count=len(frame), ordered_ids_hash=digest(frame.sample_id.tolist()))
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        if set(frames[a].sample_id) & set(frames[b].sample_id):
            raise ValueError(f'Saved IDs overlap across {a}/{b}')
        if set(frames[a].pixel_hash) & set(frames[b].pixel_hash):
            raise ValueError(f'Saved pixel hashes overlap across {a}/{b}')
    joined = pd.concat(list(frames.values()), ignore_index=True)
    if (joined.groupby('pixel_hash').label.nunique() > 1).any():
        raise ValueError('Conflicting saved labels for identical pixel hashes')
    return frames, sources


def resolve_manifest_dataset(frame, cfg):
    first = frame.relative_path.iloc[0]
    if cfg['dataset_root']:
        candidates = [Path(cfg['dataset_root'])]
    else:
        search = Path(cfg['input_root'])
        candidates = [search / 'tiny-genimage', search / 'datasets/yangsangtai/tiny-genimage']
        generator_dir = PurePosixPath(first).parts[0]
        print('[manifest] Locating dataset root using directory metadata only.', flush=True)
        candidates.extend(p.parent for p in search.rglob(generator_dir) if p.is_dir())
    matches = sorted({p.resolve() for p in candidates if (p / first).is_file()})
    if len(matches) != 1:
        raise ValueError(f'Set dataset_root to the directory containing generator folders. Candidates: '
                         f'{[str(p) for p in matches]}. No RGB/hash scan was started.')
    root = matches[0]
    resolved = frame.copy()
    paths = []
    for index, row in enumerate(resolved.itertuples()):
        path = (root / row.relative_path).resolve()
        try:
            path.relative_to(root)
        except ValueError:
            raise ValueError(f'Image path escapes dataset root: {row.relative_path}')
        if not path.is_file() or path.stat().st_size != row.file_bytes:
            raise ValueError(f'Missing or changed file (path/size check): {path}. '
                             'Use the same dataset version as the saved audit.')
        paths.append(str(path))
        if (index + 1) % 5000 == 0:
            print(f'[manifest] {index + 1:,}/{len(frame):,} file paths/sizes checked; no image decode/hash.', flush=True)
    resolved['path'] = paths  # Ignore stale absolute /kaggle/input paths from the previous run.
    return resolved, root


def load_manifest_index(cfg, out):
    root = find_manifest_root(cfg)
    source_config_path = root / 'config.json'
    if source_config_path.is_file():
        source = json.loads(source_config_path.read_text(encoding='utf-8'))
        if source.get('smoke') and not cfg['smoke']:
            raise ValueError('Cannot use smoke-only source manifests for a full training run')
        if set(source.get('train_generators', [])) != set(cfg['train_generators']):
            raise ValueError('Source config is not an All7 experiment')
        if source.get('validation_fraction') != cfg['validation_fraction']:
            raise ValueError('validation_fraction differs from saved split; reused manifests are never re-split')
    all_frames, cohorts, sources = [], {}, {}
    for seed in cfg['seeds']:
        frames, provenance = read_manifest_cohorts(root, cfg, seed)
        all_frames.extend(frames.values())
        cohorts[str(seed)] = {part: f.sample_id.tolist() for part, f in frames.items()}
        sources[str(seed)] = provenance
    merged = pd.concat(all_frames, ignore_index=True)
    # A sample may occur in several seed experiments; metadata must agree.
    for column in sorted(MANIFEST_COLUMNS - {'sample_id'}):
        if (merged.groupby('sample_id')[column].nunique(dropna=False) > 1).any():
            raise ValueError(f'Conflicting saved {column} across seeds/cohorts')
    merged = merged.drop_duplicates('sample_id', keep='first').reset_index(drop=True)
    merged, dataset = resolve_manifest_dataset(merged, cfg)
    provenance = dict(mode='reuse_manifests', manifest_root=str(root), dataset_root=str(dataset),
                      sources=sources, unique_selected_images=len(merged),
                      checks=['schema', 'ordered_split_summary_if_present', 'saved_pixel_hash_leakage',
                              'label_and_split_path_consistency', 'file_exists', 'file_byte_length'],
                      skipped=['image_decode', 'image_content_hash', 'pixel_hash_recomputation',
                               'discovery_of_unlisted_or_new_images'],
                      trust='Saved audit metadata and unchanged dataset version are trusted. '
                            'Same-size content changes are NOT detected.')
    target = out / 'manifest_reuse.json'
    if target.exists() and json.loads(target.read_text(encoding='utf-8')) != provenance:
        raise ValueError('Manifest/input fingerprint changed; choose a new output_root before resume')
    dump(target, provenance)
    save_csv(out / 'reused_manifest_index.csv', merged)
    merged.attrs['reused_cohorts'] = cohorts
    merged.attrs['manifest_root'] = str(root)
    print(f'[manifest] Reused {len(merged):,} selected images. No RGB decoding or image hashing; no re-split.', flush=True)
    return merged


def reuse_saved_splits(df, cfg, seed, out):
    cohorts = df.attrs.get('reused_cohorts', {}).get(str(seed))
    if cohorts is None:
        raise ValueError('Missing validated saved-cohort index; refusing to re-split or scan implicitly')
    indexed = df.set_index('sample_id', drop=False)
    frames = []
    for partition in MANIFEST_PARTITIONS:
        frame = indexed.loc[cohorts[partition]].reset_index(drop=True)
        frame.attrs = {}
        if cfg['smoke']:
            limit = cfg['smoke_train_per_class'] if partition == 'train' else cfg['smoke_eval_per_class']
            frame = frame[frame.groupby(['generator', 'label']).cumcount() < limit].reset_index(drop=True)
        save_csv(out / f'{partition}_manifest.csv', frame)
        frame.groupby(['generator', 'label']).size().rename('n').to_csv(out / f'{partition}_counts.csv')
        frames.append(frame)
    tr, va, te = frames
    dump(out / 'split_summary.json', dict(train=len(tr), validation=len(va), test=len(te),
         train_hash=digest(tr.sample_id.tolist()), validation_hash=digest(va.sample_id.tolist()),
         test_hash=digest(te.sample_id.tolist()), selection='Reused ordered audited All7 cohorts; no re-split',
         manifest_root=df.attrs['manifest_root'], smoke_subset=bool(cfg['smoke'])))
    print(f'[split/reused] train={len(tr):,}, validation={len(va):,}, test={len(te):,}; original ID order retained.', flush=True)
    return tr, va, te


"""All7 crop560 + label-independent blur/JPEG + fixed two-view validation.

Append after runtime.py and runtime_raw_all7.py in the standalone notebook.
The raw-only runtimes/notebooks remain unchanged.
"""
if 'GENS7' not in globals():
    from pathlib import Path
    _raw_file = Path(__file__).with_name('runtime_raw_all7.py')
    exec(compile(_raw_file.read_text(encoding='utf-8'), str(_raw_file), 'exec'), globals())

if 'load_manifest_index' not in globals():
    _manifest_file = Path(__file__).with_name('manifest_reuse.py')
    exec(compile(_manifest_file.read_text(encoding='utf-8'), str(_manifest_file), 'exec'), globals())

_raw_images = Images
_raw_validate = validate_raw_config
_raw_splits = make_splits
_raw_run = run
_full_scan = scan_dataset

VALIDATION_MONITOR = 'mean_native_fixed_processed_cross_entropy'
DEFAULT_CONFIG = dict(
    DEFAULT_CONFIG,
    output_root='/kaggle/working/gsd_all7_crop560_blur_jpeg_manifest_v2',
    data_mode='reuse_manifests', manifest_root=None,
    input_mode='center_crop_max_edge_pad14', crop_max_side=560,
    jpeg_probability=0.5, jpeg_train_quality=[75, 100], jpeg_subsampling=0,
    blur_probability=0.5, blur_sigma=[0.0, 2.0], horizontal_flip_probability=0.0,
    validation_processing_seed=42, validation_monitor=VALIDATION_MONITOR,
)


def protocol_id(cfg):
    return f"all7_center_crop{cfg['crop_max_side']}_blur_jpeg_fixedval_v2"


def validate_raw_config(cfg):
    # Reuse geometry, batch, evaluation-case and All7 safeguards, not the raw-only ban.
    _raw_validate(dict(cfg, jpeg_probability=0.0, blur_probability=0.0,
                       horizontal_flip_probability=0.0))
    if cfg['input_mode'] != 'center_crop_max_edge_pad14':
        raise ValueError('Blur/JPEG protocol requires the capped center-crop input mode')
    if cfg['horizontal_flip_probability'] != 0:
        raise ValueError('Only Gaussian blur and JPEG are enabled; flip must remain zero')
    for name in ('jpeg_probability', 'blur_probability'):
        value = cfg[name]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f'{name} must be a finite probability in [0, 1]')
    quality = cfg['jpeg_train_quality']
    if (not isinstance(quality, (list, tuple)) or len(quality) != 2 or
            any(isinstance(v, bool) or not isinstance(v, int) for v in quality) or
            not 1 <= quality[0] <= quality[1] <= 100):
        raise ValueError('jpeg_train_quality must be two ordered integers in [1, 100]')
    sigma = cfg['blur_sigma']
    if (not isinstance(sigma, (list, tuple)) or len(sigma) != 2 or
            any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in sigma) or
            not 0 <= sigma[0] <= sigma[1]):
        raise ValueError('blur_sigma must be two finite ordered nonnegative numbers')
    if isinstance(cfg['jpeg_subsampling'], bool) or cfg['jpeg_subsampling'] != 0:
        raise ValueError('This controlled protocol fixes JPEG subsampling=0 (4:4:4)')
    if isinstance(cfg['validation_processing_seed'], bool) or not isinstance(cfg['validation_processing_seed'], int):
        raise ValueError('validation_processing_seed must be an integer')
    if cfg['validation_monitor'] != VALIDATION_MONITOR:
        raise ValueError('Early stopping must monitor equal-weight native/processed validation CE')
    if cfg['data_mode'] not in ('reuse_manifests', 'full_scan'):
        raise ValueError('data_mode must be reuse_manifests or explicit full_scan')
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Need a nonempty list of distinct run seeds')


def scan_dataset(cfg, out):
    if cfg['data_mode'] == 'reuse_manifests':
        return load_manifest_index(cfg, out)
    return _full_scan(cfg, out)


def processing_parameters(sample_id, cfg, seed, train=False, epoch=0):
    """Local RNG: independent of labels, row order, workers and global random state.

    Validation deliberately ignores epoch; training changes its draws each epoch.
    The two Bernoulli gates are independent, including when an operation is disabled.
    """
    identity = ['blur_jpeg_v1', int(seed), str(sample_id)]
    identity += ['train', int(epoch)] if train else ['validation', cfg['validation_processing_seed']]
    rng = random.Random(int(digest(identity)[:16], 16))
    blur_applied = rng.random() < cfg['blur_probability']
    jpeg_applied = rng.random() < cfg['jpeg_probability']
    sigma = rng.uniform(*cfg['blur_sigma'])
    quality = rng.randint(*cfg['jpeg_train_quality'])
    return dict(blur_applied=blur_applied, blur_sigma=sigma if blur_applied else None,
                jpeg_applied=jpeg_applied, jpeg_quality=quality if jpeg_applied else None)


def apply_processing(image, parameters, cfg):
    # Operate on the decoded original, BEFORE crop/padding/normalization.
    if parameters['blur_applied']:
        image = image.filter(ImageFilter.GaussianBlur(parameters['blur_sigma']))
    if parameters['jpeg_applied']:
        image = jpeg(image, parameters['jpeg_quality'], cfg['jpeg_subsampling'])
    return image


class Images(_raw_images):
    def __init__(self, frame, cfg, seed, train=False, epoch=0, case='native'):
        super().__init__(frame, cfg, seed, train, epoch, case)
        self.seed, self.epoch = seed, epoch

    def __getitem__(self, i):
        if not self.train and self.case != 'processed':
            # Native/test cases remain byte-for-byte equivalent to the raw evaluator.
            return super().__getitem__(i)
        row = self.rows[i]
        with Image.open(row['path']) as im:
            image = im.convert('RGB')
        parameters = processing_parameters(row['sample_id'], self.cfg, self.seed,
                                           train=self.train, epoch=self.epoch)
        image = apply_processing(image, parameters, self.cfg)
        geom = input_geometry(*image.size, self.cfg)
        image = image.crop((geom['crop_left'], geom['crop_top'],
                            geom['crop_left'] + geom['crop_width'],
                            geom['crop_top'] + geom['crop_height']))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0
        h, w = x.shape[-2:]
        x = F.pad(x.unsqueeze(0), (0, (-w) % 14, 0, (-h) % 14), mode='replicate').squeeze(0)
        mean = x.new_tensor([.48145466, .4578275, .40821073])[:, None, None]
        std = x.new_tensor([.26862954, .26130258, .27577711])[:, None, None]
        return (x - mean) / std, int(row['label']), i


def save_csv(path, frame):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)


def make_splits(df, cfg, seed, out):
    if cfg['data_mode'] == 'reuse_manifests':
        train, val, tests = reuse_saved_splits(df, cfg, seed, out)
    else:
        train, val, tests = _raw_splits(df, cfg, seed, out)
    columns = [c for c in ('sample_id', 'generator', 'label', 'pixel_hash') if c in val]
    plan = pd.concat([
        val[columns].reset_index(drop=True),
        pd.DataFrame([processing_parameters(sid, cfg, seed) for sid in val.sample_id]),
    ], axis=1)
    plan['batch_id'] = np.arange(len(plan)) // cfg['test_batch']
    plan['run_seed'] = seed
    plan['validation_processing_seed'] = cfg['validation_processing_seed']
    save_csv(out / 'validation_processing_plan.csv', plan)
    dump(out / 'validation_protocol.json', dict(
        cases=['native', 'processed'], monitor=VALIDATION_MONITOR,
        weights=dict(native=0.5, processed=0.5),
        processing_order=['decode_rgb', 'blur', 'jpeg', 'center_crop', 'edge_pad14', 'normalize'],
        validation_processing_seed=cfg['validation_processing_seed'], run_seed=seed,
        plan_hash=digest(plan.to_dict('records')),
        source='validation split from train folders only; official test excluded',
        batching='same fixed ID order and membership in both validation views',
        loss='mean per-image cross entropy computed directly from logits',
    ))
    print(f'[validation] Fixed native + processed views; plan: {out / "validation_processing_plan.csv"}', flush=True)
    return train, val, tests


@torch.no_grad()
def predict(model, anchor, frame, cfg, seed, device, case, return_loss=False):
    if len(frame) == 0:
        raise ValueError('Cannot evaluate an empty cohort')
    model.eval()
    result = np.full(len(frame), np.nan)
    batches = np.full(len(frame), -1, dtype=np.int64)
    loss_sum, count = 0.0, 0
    for bid, (images, labels, indices) in enumerate(loader(frame, cfg, seed, case=case)):
        # Both branches see exactly the SAME transformed tensors; no second RNG draw.
        bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
        for start, x in image_chunks(images, cfg['micro_batch']):
            with amp(cfg, device):
                logits = model(x.to(device), bases)
            if not torch.isfinite(logits).all():
                raise FloatingPointError(f'Nonfinite {case} logits')
            ix = indices[start:start + len(x)].numpy()
            result[ix] = logits.float().softmax(-1)[:, 1].cpu().numpy()
            batches[ix] = bid
            if return_loss:
                loss_sum += float(F.cross_entropy(
                    logits.float(), labels[start:start + len(x)].to(device), reduction='sum'))
                count += len(x)
        if bid % 25 == 0:
            print(f'[{case}] batch {bid + 1}/{math.ceil(len(frame) / cfg["test_batch"])}', flush=True)
    if not np.isfinite(result).all() or (batches < 0).any():
        raise ValueError('Missing/nonfinite prediction')
    if return_loss:
        if count != len(frame) or not math.isfinite(loss_sum):
            raise FloatingPointError('Incomplete/nonfinite validation loss')
        return result, batches, loss_sum / count
    return result, batches


def evaluate_validation(model, anchor, val, cfg, seed, device):
    summaries, predictions = {}, {}
    for case in ('native', 'processed'):
        p, batches, loss = predict(model, anchor, val, cfg, seed, device, case, return_loss=True)
        summaries[case] = dict(loss=loss, **metric(val.label.to_numpy(), p, cfg['threshold']))
        predictions[case] = (p, batches)
    if not np.array_equal(predictions['native'][1], predictions['processed'][1]):
        raise ValueError('Validation batch membership changed across views')
    monitor = (summaries['native']['loss'] + summaries['processed']['loss']) / 2.0
    return monitor, summaries, predictions


def train_model(model, anchor, train, val, cfg, seed, device, folder, signature):
    groups = [dict(params=model.vision.parameters(), lr=cfg['lr']),
              dict(params=model.head.parameters(), lr=cfg['head_lr'])]
    if len(model.gsd):
        groups.append(dict(params=model.gsd.parameters(), lr=cfg['modulation_lr']))
    optimizer = torch.optim.AdamW(groups, betas=tuple(cfg['betas']), eps=cfg['eps'], weight_decay=cfg['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and cfg['precision'] == 'fp16')
    stopper = EarlyStopping(cfg['early_stopping'], cfg['patience'], cfg['min_delta'])
    latest = folder / 'last.pt'
    start, best, history = 0, float('inf'), []
    if cfg['resume'] and latest.exists():
        ck = torch.load(latest, map_location='cpu', weights_only=False)
        if ck['signature'] != signature or ck.get('protocol') != protocol_id(cfg) or ck.get('config') != cfg:
            raise ValueError('Resume protocol/config/manifests changed; use a new output_root')
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        stopper.state.update(ck['early_stopping'])
        start, best, history = ck['epoch'] + 1, ck['best'], ck['history']
        del ck
        print(f'[resume] next epoch {start + 1}, stopped={stopper.state["stopped"]}', flush=True)
    for epoch in range(start, cfg['epochs']):
        if stopper.state['stopped']:
            break
        model.train()
        running, count, skipped = 0.0, 0, 0
        tick = time.monotonic()
        for bid, (images, labels, _) in enumerate(loader(train, cfg, seed, True, epoch)):
            bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
            optimizer.zero_grad(set_to_none=True)
            for a, x in image_chunks(images, cfg['micro_batch']):
                with amp(cfg, device):
                    logits = model(x.to(device), bases)
                    loss = F.cross_entropy(logits.float(), labels[a:a + len(x)].to(device), reduction='sum') / len(images)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite training loss')
                scaler.scale(loss).backward()
                running += float(loss.detach()) * len(images)
            old = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += int(scaler.get_scale() < old)
            count += len(images)
            if bid % 10 == 0:
                print(f'[epoch {epoch + 1}] batch {bid + 1}/{math.ceil(len(train) / cfg["train_batch"])}; loss={running / count:.5f}', flush=True)
        if count == 0:
            raise ValueError('Empty training cohort')
        vloss, summaries, predictions = evaluate_validation(model, anchor, val, cfg, seed, device)
        stop = stopper.update(vloss)
        entry = dict(epoch=epoch + 1, train_loss=running / count, val_loss=vloss,
                     val_native_loss=summaries['native']['loss'], val_processed_loss=summaries['processed']['loss'],
                     validation_monitor=VALIDATION_MONITOR,
                     val_native=summaries['native'], val_processed=summaries['processed'],
                     seconds=time.monotonic() - tick, amp_skipped_steps=skipped, train_images=count,
                     modulation={l: dict(a=float(g.a.detach()), b=float(g.b.detach())) for l, g in model.gsd.items()},
                     early_stopping=dict(stopper.state))
        history.append(entry)
        print(entry, flush=True)
        state = dict(model=model.state_dict(), epoch=epoch, signature=signature, config=cfg,
                     protocol=protocol_id(cfg), validation=summaries, val_loss=vloss)
        if vloss < best:
            best = vloss
            save_checkpoint(folder / 'best.pt', state)
        save_checkpoint(latest, dict(state, optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                                     best=best, history=history, early_stopping=dict(stopper.state)))
        dump(folder / 'history.json', history)
        dump(folder / 'training_status.json', dict(epoch=epoch + 1, max_epochs=cfg['epochs'],
             validation_monitor=VALIDATION_MONITOR, val_loss=vloss, early_stopping=dict(stopper.state)))
        rows = [dict(epoch=e['epoch'], case=case, monitor_loss=e['val_loss'], **e['val_' + case])
                for e in history for case in ('native', 'processed')]
        save_csv(folder / 'validation_metrics.csv', pd.DataFrame(rows))
        columns = [c for c in ('sample_id', 'generator', 'label', 'pixel_hash') if c in val]
        frame = val[columns].copy()
        for case, (p, batches) in predictions.items():
            frame['prob_fake_' + case] = p
            frame['batch_id_' + case] = batches
        save_csv(folder / f'validation_predictions_epoch{epoch + 1:03d}.csv', frame)
        if stop:
            print('[early stopping] Restore best mean native/processed validation CE checkpoint.', flush=True)
            break
    ck = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    dump(folder / 'selected_checkpoint.json', dict(epoch=ck['epoch'] + 1,
         criterion=VALIDATION_MONITOR, val_loss=ck['val_loss'], validation=ck['validation'], signature=signature))


def run(config=None):
    cfg = dict(DEFAULT_CONFIG, **(config or {}))
    validate_raw_config(cfg)
    print('[protocol] All7 crop560; train blur/JPEG resampled per epoch, SAME tensors for anchor/detector.', flush=True)
    print('[protocol] Fixed native/processed validation; early stopping = 0.5 * CE(native) + 0.5 * CE(processed).', flush=True)
    print('[protocol] New pretrained run only; do not reuse raw-only checkpoint/output directories.', flush=True)
    print(f'[protocol] Data mode: {cfg["data_mode"]}; manifests may be reused, detector weights are NOT reused.', flush=True)
    return _raw_run(cfg)


In [ ]:
CONFIG = dict(DEFAULT_CONFIG)
CONFIG.update(
    output_root='/kaggle/working/gsd_all7_crop560_blur_jpeg_manifest_v2',
    dataset_root=None,
    data_mode='reuse_manifests', manifest_root=None,
    models=['clip_gsd'], seeds=[42],
    input_mode='center_crop_max_edge_pad14', crop_max_side=560,
    epochs=50, early_stopping=True, patience=3, min_delta=0.001,
    train_batch=8, test_batch=8, micro_batch=1, anchor_micro_batch=1,
    num_workers=0, precision='fp16', gradient_checkpointing=True,
    jpeg_probability=0.5, jpeg_train_quality=[75, 100], jpeg_subsampling=0,
    blur_probability=0.5, blur_sigma=[0.0, 2.0], horizontal_flip_probability=0.0,
    validation_processing_seed=42,
    validation_monitor='mean_native_fixed_processed_cross_entropy',
    eval_cases=['native'],
    smoke=False, resume=True,
)
# Smoke: dùng cùng crop/augmentation/validation, ít ảnh và 1 epoch; output phải riêng.
# CONFIG.update(smoke=True, output_root='/kaggle/working/gsd_all7_crop560_blur_jpeg_manifest_smoke')
# Nếu tự tìm bị nhiều ứng viên: trỏ tới thư mục chứa seed42 (không trỏ tới best.pt).
# CONFIG['manifest_root'] = '/kaggle/input/<attached-output>/gsd_raw_all7_crop560_v1'
# CONFIG['dataset_root'] = '/kaggle/input/datasets/yangsangtai/tiny-genimage'
# Audit lại từ đầu chỉ khi muốn; không tự fallback:
# CONFIG.update(data_mode='full_scan', output_root='/kaggle/working/gsd_all7_blur_jpeg_fullscan_v2')
# Test variants chỉ chạy SAU training, không dùng để chọn checkpoint:
# CONFIG['eval_cases'] = ['native', 'jpeg96', 'jpeg90', 'jpeg80']
print(json.dumps(CONFIG, indent=2))


In [ ]:
metrics = run(CONFIG)


In [ ]:
out = Path(CONFIG['output_root'])
display(pd.read_csv(out / 'macro_all.csv'))
display(pd.read_csv(out / 'metrics.csv'))
for seed in CONFIG['seeds']:
    for name in CONFIG['models']:
        folder = out / f'seed{seed}' / name
        print(folder)
        display(pd.read_csv(folder / 'validation_metrics.csv'))
        print(json.loads((folder / 'selected_checkpoint.json').read_text()))
print('Checkpoints:', list(out.glob('seed*/clip_gsd/*.pt')))
